# MCE-IRL applied workflow

This notebook estimates a delivery-route preference from demonstrated choices under known deterministic routes. It checks support, fits a normalized reward, reports trajectory-bootstrap uncertainty, evaluates held-out choices, runs a reward counterfactual, and verifies serialization.

In [1]:
import pickle
from pathlib import Path
import econirl
import jax.numpy as jnp
import numpy as np
from econirl import DeterministicTransitions, MCEIRL, MCEIRLTask, Panel, Trajectory

def make_panel(seed, n_individuals):
    rng = np.random.default_rng(seed)
    actions = rng.binomial(1, 1.0 / (1.0 + np.exp(-0.4)), size=n_individuals)
    return Panel([
        Trajectory(
            states=jnp.array([0]), actions=jnp.array([action]),
            next_states=jnp.array([1]), individual_id=individual,
            metadata={"task_id": "delivery"},
        )
        for individual, action in enumerate(actions)
    ])

checkout_root = Path.cwd().resolve().parents[1]
module_outside_checkout = not Path(econirl.__file__).resolve().is_relative_to(checkout_root)
train = make_panel(13_001, 240)
held_out = make_panel(13_002, 120)
print(f"Installed package import: {module_outside_checkout}")
print(f"Training routes: {train.num_individuals}")
print(f"Held-out routes: {held_out.num_individuals}")
print(f"Training express share: {np.asarray(train.get_all_actions()).mean():.3f}")

Installed package import: True
Training routes: 240
Held-out routes: 120
Training express share: 0.650


In [2]:
transitions = DeterministicTransitions(
    next_state=np.array([[1, 1], [1, -1]]),
    valid_action=np.array([[True, True], [True, False]]),
)
tasks = [MCEIRLTask(
    task_id="delivery", initial_state=0, terminal_states=np.array([1]), horizon=1
)]
features = np.zeros((2, 2, 1), dtype=np.float32)
features[0, 1, 0] = 1.0

model = MCEIRL(
    n_states=2, n_actions=2, discount=1.0, horizon=1,
    feature_matrix=features, feature_names=["express_preference"],
    se_method="bootstrap", n_bootstrap=19, se_seed=13_003,
)
model.fit(train, transitions=transitions, tasks=tasks)
print(f"Converged: {model.converged_}")
print(f"Termination: {model.termination_reason_}")
print(f"Bootstrap draws: {model.bootstrap_.n_successful}/{model.bootstrap_.n_requested}")

Converged: True
Termination: joint_convergence
Bootstrap draws: 19/19


In [3]:
print(model.summary())
print("\nPre-estimation diagnostics")
print(model.diagnostics_["identification"])
print(model.diagnostics_["transitions"])

Estimator
MCE-IRL (Maximum Causal Entropy feature matching)

Data
Observations: 240
Individuals: 240
State coverage: 0.500

Model
States: 2
Actions: 2
Discount factor: 1
Tasks: 1

Pre-estimation checks
Identification: identified
Action-contrast rank: 1
Transition source: compiled deterministic task views

Fit
Converged: yes
Termination: joint_convergence
Iterations: 9
Fit time: 1.220 seconds
Feature residual: 6.61866e-09

Outcome
Log likelihood: -155.387193
express_preference: 0.619039 (SE 0.183555, 95.0% CI [0.31138, 0.926003])

Uncertainty
Method: bootstrap
Confidence level: 95.0%
Bootstrap unit: individual_trajectory
Bootstrap successful draws: 19/19
Intervals: empirical percentile intervals over trajectory resamples

Limitations
Reward levels are normalized and are not identified without the supplied basis.
Transitions are treated as known inputs during reward estimation.
Counterfactual welfare in levels is withheld; policy and normalized values remain available.

Pre-estimation di

In [4]:
probabilities = model.predict_proba(np.array([0]), task_id="delivery")[0]
held_out_actions = np.asarray(held_out.get_all_actions(), dtype=int)
held_out_nll = -float(np.log(probabilities[held_out_actions]).mean())
print(f"Held-out negative log likelihood: {held_out_nll:.4f}")
print(f"Predicted express share: {probabilities[1]:.3f}")

Held-out negative log likelihood: 0.7094
Predicted express share: 0.650


In [5]:
changed_reward = model.params_["express_preference"] + 0.5
counterfactual = model.counterfactual(
    params={"express_preference": changed_reward},
    description="increase the relative reward of the express route",
)
print(counterfactual.summary(reward_level_identified=False))

                          Counterfactual Summary                          
increase the relative reward of the express route      Oracle: none
--------------------------------------------------------------------------
                              baseline  counterfactual    change
  Mean choice prob a=0           0.675           0.623    -0.052
  Mean choice prob a=1           0.325           0.377    +0.052
--------------------------------------------------------------------------
  Policy shift |dpi|:   max 0.10 . p95 0.10 . p50 0.05 . p5 0.00
  Welfare:              not identified in levels -- behavioral comparison only


In [6]:
restored = pickle.loads(pickle.dumps(model))
restored_probability = restored.predict_proba(
    np.array([0]), task_id="delivery"
)[0, 1]
print(f"Reload prediction gap: {abs(restored_probability - probabilities[1]):.3g}")
print(f"Reload summary equal: {restored.summary() == model.summary()}")

Reload prediction gap: 0
Reload summary equal: True
